In [ ]:
import pandas as pd
from pyproj import Transformer

# 1. Cargar tus datos (Suponiendo un CSV con 'lat' y 'lon')
df_puntos = pd.read_csv("../../data/lat_lon_msp_hospitales.csv")

# 2. Convertir Lat/Lon a UTM 32717 (Metros)
# EPSG:4326 es Lat/Lon WGS84 -> EPSG:32717 es tu zona UTM
transformer = Transformer.from_crs("EPSG:4326", "EPSG:32717", always_xy=True)

# Creamos las columnas X e Y en metros
df_puntos['x_utm'], df_puntos['y_utm'] = transformer.transform(df_puntos['longps'].values, df_puntos['latgps'].values)

# 3. ASIGNAR ALTURA (Interpolación rápida)
# Para que el punto "sepa" a qué altura estar, buscamos la curva de nivel más cercana
def buscar_altura_cercana(x, y, gdf_relieve):
    # Buscamos el punto más cercano en las curvas de nivel
    # Para velocidad en 1:250k, tomamos el valor del 'crv' del objeto más próximo
    distancias = gdf_relieve.distance(Point(x, y))
    idx_cercano = distancias.idxmin()
    return gdf_relieve.loc[idx_cercano, 'crv']

# Nota: Si tienes miles de puntos, es mejor usar un Spatial Index (sindex)
# Aquí lo haremos de forma simple para pocos puntos:
from shapely.geometry import Point
df_puntos['z_relieve'] = df_puntos.apply(lambda r: buscar_altura_cercana(r.x_utm, r.y_utm, gdf), axis=1)

# OPCIÓN B: Si ya conoces la altura de los puntos, solo úsala. 
# Si no, súmales un pequeño "offset" (+50m) para que no se entierren en las líneas

In [ ]:
import geopandas as gpd
import numpy as np
import plotly.graph_objects as go
import pandas as pd
from pyproj import Transformer
from scipy.spatial import cKDTree

class MapaBase3D:
    def __init__(self, archivo_shp, color_relieve='#470bf6'):
        print("Iniciando motor del mapa...")
        self.gdf = gpd.read_file(archivo_shp, engine='pyogrio')
        
        # 1. Filtro y Simplificación
        self.gdf = self.gdf[self.gdf['crv'] % 100 == 0].copy()
        self.gdf['geometry'] = self.gdf.simplify(tolerance=200, preserve_topology=True)
        
        self.min_x = self.gdf.geometry.bounds.minx.min()
        self.min_y = self.gdf.geometry.bounds.miny.min()
        self.transformer = Transformer.from_crs("epsg:4326", "epsg:32717", always_xy=True)
        
        # 2. Preparar datos para el Relieve y el Buscador (KD-Tree)
        self.x_render, self.y_render, self.z_render = [], [], []
        puntos_para_tree = []
        alturas_para_tree = []

        print("Procesando geometría...")
        for _, row in self.gdf.iterrows():
            if row.geometry.geom_type == 'LineString':
                coords = np.array(row.geometry.coords)
                n_puntos = len(coords)
                
                # Coordenadas normalizadas
                x_norm = np.round(coords[:, 0] - self.min_x, 1)
                y_norm = np.round(coords[:, 1] - self.min_y, 1)
                z_val = row['crv']

                # Lógica para Renderizado (con None para separar líneas)
                self.x_render.extend(x_norm.tolist())
                self.y_render.extend(y_norm.tolist())
                self.z_render.extend([z_val] * n_puntos) # Corregido: Creamos lista del mismo tamaño
                
                self.x_render.append(None); self.y_render.append(None); self.z_render.append(None)

                # Lógica para el Buscador (KD-Tree usa coordenadas reales)
                puntos_para_tree.append(coords[:, :2])
                alturas_para_tree.append(np.full(n_puntos, z_val))

        # 3. Crear el Árbol de búsqueda
        print("Construyendo buscador de alturas (KD-Tree)...")
        self.tree = cKDTree(np.vstack(puntos_para_tree))
        self.alturas_referencia = np.concatenate(alturas_para_tree)

        # 4. Iniciar Figura
        self.fig = go.Figure()
        self._dibujar_relieve(color_relieve)
        self._configurar_escena()

    def _dibujar_relieve(self, color):
        self.fig.add_trace(go.Scatter3d(
            x=self.x_render, y=self.y_render, z=self.z_render,
            mode='lines',
            line=dict(color=color, width=1),
            name="Relieve Base",
            hoverinfo='none'
        ))

    def _configurar_escena(self):
        # 1. Calculamos el rango real de los datos
        max_x = self.gdf.geometry.bounds.maxx.max()
        max_y = self.gdf.geometry.bounds.maxy.max()
        
        rango_x = max_x - self.min_x
        rango_y = max_y - self.min_y
        
        # 2. Establecemos la relación de aspecto
        # X siempre será 1. Y será la proporción relativa a X.
        ratio_x = 1
        ratio_y = rango_y / rango_x
        
        # AQUÍ CONTROLAS LA "ALTURA" DE LAS MONTAÑAS
        # Sube este valor (ej. 0.3 o 0.5) si quieres que se vean más altas
        ratio_z = 0.7
        self.fig.update_layout(
            template="plotly_dark",
            scene=dict(
                xaxis=dict(visible=False),
                yaxis=dict(visible=False),
                zaxis=dict(visible=False),
                aspectmode='manual',
                aspectratio=dict(x=ratio_x, y=ratio_y, z=ratio_z),
                camera=dict(
                    # 'up' determina qué dirección es "arriba" (normalmente z=1)
                    up=dict(x=0, y=0, z=1),
                    # 'center' es el punto al que la cámara mira (0,0,0 es el centro del mapa)
                    center=dict(x=6, y=6, z=0),
                    # 'eye' es la posición de la cámara. 
                    # Aumenta 'z' para alejarte (ver más área)
                    # X e Y en 1.25 dan una vista diagonal equilibrada
                    eye=dict(x=1.25, y=1.25, z=1.5) 
                )
            ),
            paper_bgcolor='black',
            margin=dict(l=0, r=0, b=0, t=0),
            showlegend=True
        )

    def añadir_puntos(self, df, col_lat, col_lon, col_info, color='#f81d78', size=6, offset=30):
        # Proyectar a UTM
        x_utm, y_utm = self.transformer.transform(df[col_lon].values, df[col_lat].values)
        
        # Buscar altura en el KD-Tree
        distancias, indices = self.tree.query(np.column_stack((x_utm, y_utm)))
        z_relieve = self.alturas_referencia[indices]
        
        self.fig.add_trace(go.Scatter3d(
            x=x_utm - self.min_x,
            y=y_utm - self.min_y,
            z=z_relieve + offset,
            mode='markers',
            marker=dict(size=size, color=color, symbol='diamond'),
            text=df[col_info],
            hoverinfo='text',
            name=f"Capa: {col_info}"
        ))
        print(f"Puntos añadidos con éxito.")

    def mostrar(self, nombre_archivo="mapa_final.html"):
        self.fig.write_html(nombre_archivo, include_plotlyjs='cdn')
        print(f"Mapa guardado en: {nombre_archivo}")

In [ ]:
# 1. Crear el mapa base
mapa = MapaBase3D("../../maps/curva_nivel_l_menos_preciso/curva_nivel_l.shp")

# 2. Cargar tus datos (CSV, Excel o DataFrame existente)
df_datos = pd.read_csv("../../data/lat_lon_msp_hospitales.csv") 
# Supongamos que df_datos tiene columnas: ['latitud', 'longitud', 'nombre_estacion', 'tipo']
# 1. Cargar tus datos (Suponiendo un CSV con 'lat' y 'lon')
# 3. Graficar los puntos
# Pasamos el DataFrame, el nombre de la col de lat, la de lon y la que queremos ver (nombre)
mapa.añadir_puntos(
    df=df_datos, 
    col_lat='latgps', 
    col_lon='longps', 
    col_info='nombre_centro_de_salud', 
    color='#c3fc0d' # Verde neón
)

# 4. Ver resultado
mapa.mostrar("mi_proyecto_3d.html")

In [ ]:
def obtener_color_personalizado(altitud):
    if altitud < 500: 
        return '#470bf6'   # Azul profundo (Valles/Base)
    elif altitud < 1200: 
        return '#63ede0'   # Cian (Laderas bajas)
    elif altitud < 2200: 
        return '#c3fc0d'   # Verde Neón (Zonas medias)
    elif altitud < 3200: 
        return '#fd6c1d'   # Naranja (Altas montañas)
    else: 
        return '#f81d78'   # Fucsia (Cumbres máximas)